# Spending Analytics: End-to-End Data Mining Pipeline

This notebook demonstrates the full project workflow:

1. Generate one consolidated synthetic banking dataset
2. Explore the dataset
3. Prepare features for machine learning
4. Train and evaluate the transaction category classifier
5. Save `spending_classifier.pkl`
6. Load transactions into SQLite
7. Test chatbot SQL examples

The Streamlit app uses the same files and architecture demonstrated here.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

from paths import DATASET_PATH, MODEL_PATH, DATABASE_PATH
from generate_synthetic_dataset import generate_transactions, CATEGORY_TAXONOMY
from data_utils import add_model_features, load_dataset
from database import save_transactions, load_dataframe, validate_select_sql
from retrain_unified_model import train_model
from llm_client import LLMSettings, generate_sql, answer_question

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Dataset path:", DATASET_PATH)
print("Model path:", MODEL_PATH)
print("Database path:", DATABASE_PATH)

## 1. Generate Synthetic Dataset

The generator is deterministic and creates transactions for Student, Professional, and Family profiles across RBC, TD, and Scotiabank for 2024-2026.

In [ ]:
df_generated = generate_transactions()
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
df_generated.to_csv(DATASET_PATH, index=False)

print(f"Generated {len(df_generated):,} transactions")
df_generated.head()

## 2. Validate Category Taxonomy

The project uses a clean final set of categories for the presentation and model.

In [ ]:
print("Final taxonomy:")
for category in CATEGORY_TAXONOMY:
    print("-", category)

observed_categories = sorted(df_generated["category"].unique())
unexpected = sorted(set(observed_categories) - set(CATEGORY_TAXONOMY))
missing = sorted(set(CATEGORY_TAXONOMY) - set(observed_categories))

print("\nObserved categories:", observed_categories)
print("Unexpected categories:", unexpected)
print("Missing categories:", missing)

## 3. Exploratory Data Analysis

Basic checks show data volume by profile, bank, year, and category.

In [ ]:
df = load_dataset()
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.to_period("M").astype(str)

print(df.info())
df.describe(include="all")

In [ ]:
display(df.groupby(["profile", "bank"]).size().rename("records").reset_index())
display(df.groupby("year").size().rename("records").reset_index())
display(df.groupby("category").agg(records=("category", "size"), total_amount=("amount", "sum")).reset_index())

## 3.1 Visual Exploratory Analysis

These charts summarize the main spending patterns for the presentation.

In [ ]:
import plotly.express as px

expense_df = df[df["amount"] < 0].copy()
expense_df["expense_amount"] = expense_df["amount"].abs()
cashflow_df = df.copy()
cashflow_df["expense_amount"] = cashflow_df["amount"].where(cashflow_df["amount"] < 0, 0).abs()
cashflow_df["income_amount"] = cashflow_df["amount"].where(cashflow_df["amount"] > 0, 0)

category_spending = expense_df.groupby("category", as_index=False)["expense_amount"].sum().sort_values("expense_amount", ascending=False)

fig = px.bar(
    category_spending,
    x="category",
    y="expense_amount",
    title="Spending by Category",
    labels={"category": "Category", "expense_amount": "Total Spending (CAD)"},
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

In [ ]:
monthly_spending = expense_df.groupby("month", as_index=False)["expense_amount"].sum()

fig = px.line(
    monthly_spending,
    x="month",
    y="expense_amount",
    markers=True,
    title="Monthly Spending Trend",
    labels={"month": "Month", "expense_amount": "Total Spending (CAD)"},
)
fig.show()

In [ ]:
profile_bank_counts = df.groupby(["profile", "bank"], as_index=False).size().rename(columns={"size": "transaction_count"})

fig = px.bar(
    profile_bank_counts,
    x="profile",
    y="transaction_count",
    color="bank",
    barmode="group",
    title="Transactions by Profile and Bank",
    labels={"profile": "Profile", "transaction_count": "Transactions", "bank": "Bank"},
)
fig.show()

In [ ]:
monthly_cashflow = cashflow_df.groupby("month", as_index=False).agg(
    income=("income_amount", "sum"),
    expenses=("expense_amount", "sum"),
)
monthly_cashflow_long = monthly_cashflow.melt(id_vars="month", value_vars=["income", "expenses"], var_name="type", value_name="amount")

fig = px.line(
    monthly_cashflow_long,
    x="month",
    y="amount",
    color="type",
    markers=True,
    title="Income vs Expenses by Month",
    labels={"month": "Month", "amount": "Amount (CAD)", "type": "Cash Flow Type"},
)
fig.show()

## 4. Feature Engineering

The model uses text, numeric, and categorical features:

- TF-IDF text from merchant, description, profile, and bank
- Numeric features: amount, absolute amount, month, day of week, income flag
- Categorical features: profile, bank, currency

In [ ]:
features = add_model_features(df)
features[["date", "merchant", "description", "amount", "category", "profile", "bank", "month", "day_of_week", "is_income", "abs_amount", "text"]].head()

## 5. Train and Save the Unified Model

This calls the same training function used by `retrain_unified_model.py`. The saved artifact is `spending_classifier.pkl`.

In [ ]:
model_package = train_model(DATASET_PATH, MODEL_PATH)
print("Saved model to:", MODEL_PATH)
print(model_package["classification_report"])

## 6. Load Transactions into SQLite

The Streamlit app uses the same SQLite database. Uploads can replace or append records; this notebook resets the database from the generated dataset.

In [ ]:
records_loaded = save_transactions(load_dataset(), mode="replace")
db_df = load_dataframe()

print(f"Loaded {records_loaded:,} records into SQLite")
print("Database rows:", len(db_df))
db_df.head()

In [ ]:
with sqlite3.connect(DATABASE_PATH) as conn:
    summary = pd.read_sql_query(
        """
        SELECT profile, bank, COUNT(*) AS records,
               ROUND(SUM(CASE WHEN amount < 0 THEN ABS(amount) ELSE 0 END), 2) AS spending,
               ROUND(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 2) AS income
        FROM transactions
        GROUP BY profile, bank
        ORDER BY profile, bank;
        """,
        conn,
    )

summary

## 7. Chatbot SQL Examples

The chatbot can use Ollama or Hugging Face, but the deterministic heuristic fallback is useful for testing and demos. Each generated query is validated as SELECT-only before execution.

In [ ]:
settings = LLMSettings(provider="Heuristic")
questions = [
    "How much did I spend on transportation?",
    "Show my Starbucks transactions.",
    "What did I spend in March 2025?",
    "What are my top merchants?",
    "How much income did I receive in 2026?",
    "Compare January and February spending.",
    "What is my largest expense category?",
    "How many transactions do I have?",
    "What did the Family profile spend on groceries?",
    "How much did I spend with TD?",
]

rows = []
for question in questions:
    sql = generate_sql(question, settings, selected_profile="All")
    validate_select_sql(sql)
    answer, executed_sql, result = answer_question(question, settings, selected_profile="All")
    rows.append({"question": question, "answer": answer, "rows_returned": len(result), "sql": executed_sql})

pd.DataFrame(rows)

## 8. End-to-End Summary

At this point the project has:

- A consolidated generated dataset at `synthetic_bank_data/combined_transactions.csv`
- A trained classifier at `spending_classifier.pkl`
- A populated SQLite database at `spending_analytics.db`
- Valid chatbot SQL examples ready for the Streamlit app

Run the dashboard with:

```bash
streamlit run app.py
```

## 9. Optional Demo Dataset

The deterministic dataset above is the grading and retraining dataset. For presentations, the project also includes `generate_demo_dataset.py`, which writes randomized data to `synthetic_bank_data/demo_transactions.csv` without changing `combined_transactions.csv`.

In [ ]:
from paths import DEMO_DATASET_PATH
from generate_demo_dataset import generate_demo_transactions

demo_df = generate_demo_transactions(
    profiles=["Student", "Professional", "Family"],
    banks=["RBC", "TD", "Scotiabank"],
    years=[2024, 2025, 2026],
    transactions_per_profile=500,
    seed=None,
)
demo_df.to_csv(DEMO_DATASET_PATH, index=False)

print(f"Generated {len(demo_df):,} demo transactions")
print("Saved to:", DEMO_DATASET_PATH)
demo_df.head()